# OmniVoice Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/k2-fsa/OmniVoice/blob/master/docs/OmniVoice.ipynb)

This notebook demonstrates the basic usage of [OmniVoice](https://github.com/k2-fsa/OmniVoice), a massively multilingual zero-shot TTS model supporting 600+ languages.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI, no code needed)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice

## 1. Installation

Colab already provides a compatible PyTorch + CUDA environment, so we only need to install OmniVoice.

In [4]:
!pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.6 MB/s eta 0:00:00


## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

In [5]:
from omnivoice import OmniVoice
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
    load_asr=True,
)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

### 3.2 Voice Cloning

Clone a voice from a short (3-10s) reference audio clip. Upload your own `ref.wav` or use any audio file.

`ref_text` is optional — if omitted, the model uses Whisper ASR to auto-transcribe it.

In [7]:
import re


def clean_markdown_for_tts(text):
    # Xóa ký hiệu tiêu đề Markdown
    text = re.sub(r"(?m)^\s*#{1,6}\s*", "", text)

    # Xóa ký hiệu in đậm, in nghiêng
    text = text.replace("**", "")
    text = text.replace("__", "")

    # Chuyển đường phân cách thành một lần xuống đoạn
    text = re.sub(
        r"(?m)^\s*[-*_]{3,}\s*$",
        "\n\n",
        text,
    )

    # Xóa các cue tiếng Việt trong dấu ngoặc vuông
    text = re.sub(
        r"\[(hắng giọng|thì thầm|cười khẩy|đập bàn)\]",
        "",
        text,
        flags=re.IGNORECASE,
    )

    # Chuyển tag được OmniVoice hỗ trợ
    text = re.sub(
        r"\[thở dài\]",
        "[sigh]",
        text,
        flags=re.IGNORECASE,
    )

    # Không để quá nhiều dấu ba chấm
    text = re.sub(r"\.{4,}", "...", text)

    # Không để quá hai dòng trống
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Xóa khoảng trắng thừa
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


In [1]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

Upload a reference audio file (wav/mp3/flac):


Saving ElevenLabs_2026-07-19T16_11_39_Tuan - Friendly, Calm and Trustworthy_pvc_sp89_s85_sb47_v3.mp3 to ElevenLabs_2026-07-19T16_11_39_Tuan - Friendly, Calm and Trustworthy_pvc_sp89_s85_sb47_v3.mp3
Uploaded: ElevenLabs_2026-07-19T16_11_39_Tuan - Friendly, Calm and Trustworthy_pvc_sp89_s85_sb47_v3.mp3


In [6]:
from pathlib import Path

print("Upload file")
uploaded = files.upload()
script_path = next(iter(uploaded.keys()))
script_text = Path(script_path).read_text(
    encoding="utf-16"
).strip()
script_text = clean_markdown_for_tts(script_text)
# print(f"Uploaded: {script_text}")
audio = model.generate(
    text=script_text,
    ref_audio=ref_audio_path,
    num_step=32,
    speed=1.03,
    # Tăng độ dài mỗi đoạn từ mặc định 15 giây lên 25 giây
    audio_chunk_duration=25.0,
    audio_chunk_threshold=30.0,

    # Giảm khoảng im lặng tự thêm ở đầu và cuối mỗi đoạn
    pad_duration=0.03,
    fade_duration=0.03,

    postprocess_output=True,
)

sf.write("clone_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes: gender, age, pitch, style (whisper), English accent, Chinese dialect. See [docs/voice-design.md](https://github.com/k2-fsa/OmniVoice/blob/master/docs/voice-design.md) for the full list.

In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))

### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], 24000)
display(Audio(audio[0], rate=24000))